[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/echochoho1010/Echo-capstone/blob/main/codebook/pro_network_n_minus_1.ipynb)

This copy loads the scenario CSVs from GitHub, so it runs in Colab without the local folders. The files are `lane_summary.csv` and `path_summary.csv` under `data/pro-network/Baseline/` and each `data/pro-network/{SC} down/` directory. They sit beside the East Coast scenarios, not inside them: several hub names are shared, and the two runs are different networks.

# Pro network N-1

This is the north east demo (`northeastdemo`), the network from the "check pro" run. Baseline, then each of the six sort centers taken offline on its own. Positions are the same in every panel.

Sort centers, north to south: BOSC, NJSC, HASC, CBSC, CLTC, ATLS. The measured window is 14 days, 2026-05-04 inclusive to 2026-05-18 exclusive.

- A line is one facility connection, both directions combined
- Color is daily package flow over that measured window
- A gray label has no active connection in that scenario
- Facility bars are packages in plus packages out per day

Hub links in this network: BOSC–NJSC, NJSC–HASC, HASC–CBSC, HASC–CLTC, CBSC–CLTC, CBSC–ATLS, CLTC–ATLS.

## Two kinds of loss

**Last-mile, no alternate.** The destination branch is reached only through this sort center. Boston's branches hang off BOSC, the New Jersey and Philadelphia branches off NJSC, Harrisburg's branches (including Baltimore, DC, Pittsburgh, and Buffalo) off HASC, Columbus / Cleveland / Cincinnati off CBSC, the Charlotte branches off CLTC, and the Atlanta branches off ATLS. With that center offline, those packages have no inbound lane and are not delivered.

**Through traffic.** The destination is somewhere else, and the baseline path went through this sort center. The router delivers them on another path only when that path still reaches the destination. Here that happens for CLTC: Atlanta–Harrisburg traffic moves via CBSC and still arrives. For the other five centers, the offline center is the only hop that reaches those destinations, so the packages are routing drops. The last section measures the split, the recovered volume, and the extra linehaul and transit time on the recovered packages.


In [ ]:
from urllib.parse import quote

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import Normalize

# Scenario folders live under data/pro-network/: Baseline/ and "{SC} down/".
DATA_BASE = "https://raw.githubusercontent.com/echochoho1010/Echo-capstone/main/data/pro-network"


def data_csv(scenario, filename):
    return f"{DATA_BASE}/{quote(scenario)}/{filename}"


# North to south. Display order only.
SCS = ["BOSC", "NJSC", "HASC", "CBSC", "CLTC", "ATLS"]
SCENARIOS = ["Baseline"] + [f"{code} down" for code in SCS]

raw = {name: pd.read_csv(data_csv(name, "lane_summary.csv")) for name in SCENARIOS}
for name, frame in raw.items():
    print(f"{name}: {len(frame):,} lane rows")


In [ ]:
MEASURED_DAYS = 14
HUBS = SCS


def undirected_daily(frame):
    """Packages per day on each facility pair, both directions combined."""
    data = frame.copy()
    data["day"] = pd.to_datetime(data["day"])
    window = data[(data["day"] >= "2026-05-04") & (data["day"] < "2026-05-18")]
    lanes = window.groupby(["origin", "dest"], as_index=False)["pkgs"].sum()
    lanes["left"] = lanes[["origin", "dest"]].min(axis=1)
    lanes["right"] = lanes[["origin", "dest"]].max(axis=1)
    edges = lanes.groupby(["left", "right"], as_index=False)["pkgs"].sum()
    edges["daily"] = edges["pkgs"] / MEASURED_DAYS
    return edges.loc[edges["daily"] > 0, ["left", "right", "daily"]].reset_index(drop=True)


def grouped_positions(edges):
    """Place each branch beside the sort center it uses most.

    Purely for visual layout, not a network property.
    """
    adjacent = {}
    for row in edges.itertuples(index=False):
        adjacent.setdefault(row.left, []).append((row.right, row.daily))
        adjacent.setdefault(row.right, []).append((row.left, row.daily))

    assigned = {hub: hub for hub in HUBS}
    pending = set(adjacent) - set(assigned)
    for node in list(pending):
        choices = [(hub, weight) for hub, weight in adjacent[node] if hub in HUBS]
        if choices:
            assigned[node] = max(choices, key=lambda item: item[1])[0]
            pending.remove(node)
    while pending:
        progressed = False
        for node in list(pending):
            choices = [
                (assigned[other], weight)
                for other, weight in adjacent[node]
                if other in assigned
            ]
            if choices:
                assigned[node] = max(choices, key=lambda item: item[1])[0]
                pending.remove(node)
                progressed = True
        if not progressed:
            break

    positions, angles = {}, {}
    gap = 0.42
    slot = (2 * np.pi - gap * len(HUBS)) / len(assigned)
    angle = -np.pi / 2
    for hub in HUBS:
        members = [hub] + sorted(node for node, group in assigned.items() if group == hub and node != hub)
        for node in members:
            positions[node] = np.array([np.cos(angle), np.sin(angle)])
            angles[node] = angle
            angle += slot
        angle += gap
    return positions, angles


flows = {name: undirected_daily(frame) for name, frame in raw.items()}
combined = (
    pd.concat(flows.values())
    .groupby(["left", "right"], as_index=False)["daily"]
    .max()
)
positions, angles = grouped_positions(combined)
assert set(HUBS) <= set(positions)


In [ ]:
rows = []
for name, edges in flows.items():
    rows.append({
        "Scenario": name,
        "Active connections": len(edges),
        "Packages per day": edges["daily"].sum(),
    })
summary = pd.DataFrame(rows).set_index("Scenario")
print(summary.to_string(float_format=lambda value: f"{value:,.0f}"))


In [ ]:
peak_daily = max(edges["daily"].max() for edges in flows.values())
norm = Normalize(vmin=0, vmax=peak_daily)
cmap = plt.cm.viridis


def draw_curve(ax, start, end, color):
    midpoint = (start + end) / 2
    span = np.linalg.norm(end - start)
    control = midpoint * (0.08 + 0.62 * min(span / 2, 1))
    t = np.linspace(0, 1, 40)[:, None]
    curve = (1 - t) ** 2 * start + 2 * (1 - t) * t * control + t ** 2 * end
    ax.plot(curve[:, 0], curve[:, 1], color=color, lw=1.8, solid_capstyle="round", zorder=1)


def draw_network(ax, edges, title):
    connected = set(edges["left"]) | set(edges["right"])
    for row in edges.sort_values("daily").itertuples(index=False):
        draw_curve(ax, positions[row.left], positions[row.right], cmap(norm(row.daily)))
    for node, point in positions.items():
        hub = node in HUBS
        ax.scatter(
            [point[0]], [point[1]],
            s=42 if hub else 16,
            c="#111827" if node in connected else "#d1d5db",
            zorder=2,
        )
        angle = angles[node]
        rotation = np.degrees(angle)
        alignment = "left"
        if np.cos(angle) < 0:
            rotation += 180
            alignment = "right"
        ax.text(
            np.cos(angle) * 1.18,
            np.sin(angle) * 1.18,
            node,
            rotation=rotation,
            rotation_mode="anchor",
            ha=alignment,
            va="center",
            fontsize=6,
            weight="bold" if hub else "regular",
            color="#111827" if node in connected else "#9ca3af",
        )
    ax.set_title(title, fontsize=12, weight="bold")
    ax.set_xlim(-1.7, 1.7)
    ax.set_ylim(-1.7, 1.7)
    ax.set_aspect("equal")
    ax.axis("off")


fig, axes = plt.subplots(2, 4, figsize=(18, 10), constrained_layout=True)
flat = axes.ravel()
for ax, name in zip(flat, SCENARIOS):
    draw_network(ax, flows[name], name)
for ax in flat[len(SCENARIOS):]:
    ax.axis("off")
colorbar = fig.colorbar(
    plt.cm.ScalarMappable(norm=norm, cmap=cmap), ax=axes, shrink=0.6, pad=0.01
)
colorbar.set_label("Packages per day")
fig.suptitle("Daily package flow with each sort center offline", fontsize=16, weight="bold")
plt.show()


In [ ]:
def facility_daily(frame):
    """Packages per day touching a facility, inbound plus outbound."""
    data = frame.copy()
    data["day"] = pd.to_datetime(data["day"])
    window = data[(data["day"] >= "2026-05-04") & (data["day"] < "2026-05-18")]
    inbound = window.groupby("dest")["pkgs"].sum()
    outbound = window.groupby("origin")["pkgs"].sum()
    return inbound.add(outbound, fill_value=0) / MEASURED_DAYS


baseline_flow = facility_daily(raw["Baseline"])
facility_order = sorted(positions, key=lambda node: angles[node])

for code in SCS:
    compared = pd.DataFrame({
        "Baseline": baseline_flow.reindex(facility_order, fill_value=0),
        f"{code} down": facility_daily(raw[f"{code} down"]).reindex(facility_order, fill_value=0),
    })
    fig, ax = plt.subplots(figsize=(16, 4.8), constrained_layout=True)
    x = np.arange(len(facility_order))
    width = 0.42
    ax.bar(x - width / 2, compared["Baseline"], width, label="Baseline", color="#111827")
    ax.bar(x + width / 2, compared[f"{code} down"], width, label=f"{code} down", color="#d97706")
    ax.set_xticks(x)
    ax.set_xticklabels(facility_order, rotation=90)
    for label in ax.get_xticklabels():
        if label.get_text() in HUBS:
            label.set_fontweight("bold")
    ax.set_ylabel("Packages per day")
    ax.set_title(f"Facility flow: Baseline vs {code} down")
    ax.legend()
    plt.show()


## Corridor links

The seven hub-to-hub links that exist in this network. A dark cell is a link cut by that outage. A brighter cell is not, by itself, packages that found another path. The split below is the measurement.


In [ ]:
# Hub links that exist, ordered roughly north to south.
CORRIDOR = [
    ("BOSC", "NJSC"),
    ("NJSC", "HASC"),
    ("HASC", "CBSC"),
    ("HASC", "CLTC"),
    ("CBSC", "CLTC"),
    ("CBSC", "ATLS"),
    ("CLTC", "ATLS"),
]

def edge_daily(edges, a, b):
    left, right = sorted((a, b))
    hit = edges[(edges["left"] == left) & (edges["right"] == right)]
    return float(hit["daily"].sum()) if len(hit) else 0.0


corridor = pd.DataFrame(
    [[edge_daily(flows[name], a, b) for a, b in CORRIDOR] for name in SCENARIOS],
    index=SCENARIOS,
    columns=[f"{a}–{b}" for a, b in CORRIDOR],
)
fig, ax = plt.subplots(figsize=(11, 4.2), constrained_layout=True)
image = ax.imshow(corridor.to_numpy(), aspect="auto", cmap="YlOrRd")
ax.set_xticks(range(len(corridor.columns)), corridor.columns, rotation=30, ha="right")
ax.set_yticks(range(len(corridor.index)), corridor.index)
ax.set_title("Sort-center corridor flow")
fig.colorbar(image, ax=ax, label="Packages per day")
plt.show()


## Measured split

Last-mile packages per day, through-traffic packages per day, and packages delivered on a different path. Extra linehaul and extra transit hours are counted only on the recovered packages.


In [ ]:
paths = {name: pd.read_csv(data_csv(name, "path_summary.csv")) for name in SCENARIOS}
baseline_paths = paths["Baseline"]


def inbound_lanes(frame):
    data = frame.copy()
    data["day"] = pd.to_datetime(data["day"])
    window = data[(data["day"] >= "2026-05-04") & (data["day"] < "2026-05-18")]
    totals = window.groupby(["origin", "dest"])["pkgs"].sum()
    inbound = {}
    for (origin, dest), pkgs in totals.items():
        if pkgs > 0:
            inbound.setdefault(dest, set()).add(origin)
    return inbound


def exclusive_subtree(sc, inbound):
    """Branches reached only by a handoff that dies with this sort center."""
    owned = set()
    changed = True
    while changed:
        changed = False
        for node, sources in inbound.items():
            if node in SCS or node in owned:
                continue
            if sources and sources <= ({sc} | owned):
                owned.add(node)
                changed = True
    return owned


inbound = inbound_lanes(raw["Baseline"])
loss_rows = []
branch_rows = []
for code in SCS:
    owned = exclusive_subtree(code, inbound)
    compared = baseline_paths.merge(
        paths[f"{code} down"], on=["origin", "final_dest"], suffixes=("_base", "_down")
    )
    used = compared[compared["path_base"].str.split(">").apply(lambda nodes: code in nodes)]
    local = compared[compared["final_dest"].isin(owned)]
    through = used[~used["final_dest"].isin(owned | {code})]
    recovered = used[(used["delivered_down"] > 0) & (used["path_base"] != used["path_down"])]
    last_mile_lost = (local["delivered_base"] - local["delivered_down"]).clip(lower=0)
    through_lost = (through["delivered_base"] - through["delivered_down"]).clip(lower=0)

    def extra(frame, value, weight):
        if frame.empty:
            return 0.0
        base = frame[f"{value}_base"] / frame[f"{weight}_base"]
        down = frame[f"{value}_down"] / frame[f"{weight}_down"]
        return float(((down - base) * frame[f"{weight}_down"]).sum())

    loss_rows.append({
        "Outage": code,
        "Last-mile lost": last_mile_lost.sum() / MEASURED_DAYS,
        "Through traffic lost": through_lost.sum() / MEASURED_DAYS,
        "Recovered on another path": recovered["delivered_down"].sum() / MEASURED_DAYS,
        "Extra linehaul and touch $": extra(recovered, "cost_cents", "pkgs") / 100 / MEASURED_DAYS,
        "Extra transit hours": extra(recovered, "transit_ticks", "delivered") * 0.25 / MEASURED_DAYS,
    })
    per_branch = last_mile_lost.groupby(local["final_dest"]).sum() / MEASURED_DAYS
    for dest, daily in per_branch.items():
        if daily > 0:
            branch_rows.append({"sc": code, "branch": dest, "daily": float(daily)})

loss = pd.DataFrame(loss_rows).set_index("Outage")
branches = pd.DataFrame(branch_rows)
print(loss.to_string(float_format=lambda value: f"{value:,.1f}"))
print()
print(branches.to_string(index=False, float_format=lambda value: f"{value:,.1f}"))


In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.8), constrained_layout=True)
x = np.arange(len(loss))
ax.bar(x, loss["Last-mile lost"], color="#111827", label="Last-mile, no alternate")
ax.bar(
    x,
    loss["Through traffic lost"],
    bottom=loss["Last-mile lost"],
    color="#d97706",
    label="Through traffic, no alternate",
)
ax.set_xticks(x, loss.index)
ax.set_ylabel("Packages per day not delivered")
ax.set_title("What each sort-center outage actually loses")
ax.legend()
plt.show()

colors = {
    "BOSC": "#1e3a5f",
    "NJSC": "#b45309",
    "HASC": "#0f766e",
    "CBSC": "#6d28d9",
    "CLTC": "#0369a1",
    "ATLS": "#3f6212",
}
ordered = branches.sort_values(["sc", "daily"], ascending=[True, True])
fig, ax = plt.subplots(figsize=(10, 7.2), constrained_layout=True)
y = np.arange(len(ordered))
ax.barh(y, ordered["daily"], color=[colors[code] for code in ordered["sc"]])
ax.set_yticks(y, [f"{branch}   {sc}" for sc, branch in zip(ordered["sc"], ordered["branch"])])
ax.set_xlabel("Packages per day not delivered")
ax.set_title("Last-mile branches with no handoff around the offline sort center")
plt.show()
